In [1]:
import sys
import os
from pathlib import Path

import sqlglot
import yaml

from src.core.ext_parser import ExtParser
from src.core.parser import HiveScriptParser
from src.transformers.raw_pyspark_transformer import RawPySparkTransformer
from src.jinja.environment import render_template
from src.paths import *

In [2]:
from utils.file_utils import parse_file_name

# Test with the k2_bank DDL file
# script_name = "raw_k2_bank"
# script_name = "raw_general_reference_lookup"
# script_name = "raw_lmskibb2_tbl_account"
# script_name = "raw_mhbos_m_client_crs"
# script_name = "raw_fra_connected_parties"
script_name = "cur_dim_contact"

layer, sub_layer, source_name, base_table = parse_file_name(script_name)

# script_name = "com_t_mhbos_m_client"
datalake_type_subfolder = 'dml'
datalake_layer_subfolder = script_name.split("_")[0]

# sql_file_path = PROJECT_ROOT / "output" / "input" / "ddl" / "raw" / f"{script_name}.sql"
sql_file_path = DATALAKE_SCRIPT_DIR / datalake_type_subfolder / datalake_layer_subfolder / f"{script_name}.sql"
ext_file_path = DATALAKE_SCRIPT_DIR / "ext" / "xml" / f"{source_name}_{base_table}.xml"


output_file_path = PROJECT_ROOT / "output" / "migration" / datalake_type_subfolder / datalake_layer_subfolder / f"{script_name}.py"

variable_path = VARIABLE_CONFIG_PATH


output_file_path = PROJECT_ROOT / "output" / "migration" / datalake_type_subfolder / datalake_layer_subfolder / f"{script_name}.py"

In [3]:
print(f"[1] Parsing file {sql_file_path.name}...")
context = HiveScriptParser.parse_file(str(sql_file_path))
# ext_parser = ExtParser()
# context.ext_context = ext_parser.parse_ext(ext_file_path)

print("[2] Initializing Optimized Transformer...")
transformer = RawPySparkTransformer(config_root=PROJECT_ROOT / "configs")
render_model = transformer.transform(context)

print("[3] Rendering Template (optimized_pyspark.jinja)...")
final_script = render_template(
    template_name="pyspark/optimized_pyspark.jinja",
    render_model=render_model
)

# 2. Load mapping configuration from YAML
output_file_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_file_path, 'w', encoding='utf-8') as f:
    f.write(final_script)

print("\n" + "=" * 50)
print("OPTIMIZED PYTHON FILE RESULT")
print("=" * 50 + "\n")
print(final_script)


[1] Parsing file cur_dim_contact.sql...
[2] Initializing Optimized Transformer...
Rule drop_partition triggered for <class 'sqlglot.expressions.Alter'>
[3] Rendering Template (optimized_pyspark.jinja)...

OPTIMIZED PYTHON FILE RESULT

"""
Purpose:    curated - Snapshot table script
Author:     Sunline
Usage:      python $ETL_HOME/script/main.py yyyymmdd [file_name]
CreateDate: 2023-08-18 00:00:00
FileType:   DML
Logs:
Table name: DIM_CONTACT
Table comment: DIM_CONTACT
Creation date: 2023-08-18 00:00:00
Primary key field: OWNER_ID,CONTACT_OWNER_TYPE,CONTACT_TYPE
Attribution hierarchy: curated
Attribution subject: cust
Main application: None
Analyst: zhairuoping
Time granularity: None
Retention period: None
Descriptive information: None
log:  chenguanhong  20240731    add smf/sbl/lms source (uat)
log:  davidyip      20230903    add T1.CLEAN_RULE_FLAG filter for lms and sbl (uat)
log:  marcoong      20250128    add agent_assistant from MHBOS
log:  marcoong      20250326    add new source:

In [4]:
ext_parser.parse_ext(ext_file_path)

NameError: name 'ext_parser' is not defined

In [ ]:

rule_file_path = PROJECT_ROOT / "configs" / "rules" / "optimizations" / "k2.yaml"

with open(rule_file_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)



# for rule in config["rules"]:
#     if not rule.get("enabled", False):
#         continue
#
#     if self._is_rule_triggered(rule, context):
#         action_type = rule.get("action", {}).get("type")
#         handler = action_handlers.get(action_type)
#
#         if handler:
#             # A rule was triggered and a handler exists, stop processing more rules.
#             return handler(rule, node, context)
#
#         # Stop at the first triggered rule, even if the action is unknown.
#         return None

for rule_name, rule_param in config["rules"].items():
    print(rule_param.get("enabled", False))